## Projeto Polars
### Análise base de dados Varejo e Atributos Outbox 

#### Metodologia CRISP-DM

**Entendimento do negócio** <br>
**Entendimento dos dados** (Estamos aqui) <br>
**Preparação dos dados** (Desejamos finalizar hoje)<br>
**Modelagem** <br>
**Avaliação** <br>
**Implementação** <br>

#### Dependências

In [6]:
from pathlib import Path
from datetime import datetime, date, timedelta

import polars as pl
import polars.selectors as cs
from polars.datatypes import UInt16, Utf8, Float32, Date
from polars.exceptions import (
    SchemaError,
    ComputeError,
    ColumnNotFoundError,
    PolarsError,
)

import duckdb as dd


#### Estrutura de arquivos
```
RaizProjeto
|____ data
     |____ raw
     |____ staging
           |____ base_amostrada.parquet
           |____ base_tratada.parquet
     |____ prod
           |____ modelo
           |____ base_premodelagem.parquet
|____ doc
     | requisitos.txt
     | metricas.txt
|____ src
     |____ config
           |____ caminhos.yaml
     |____ constantes
           |____ safras.yaml
     |____ tratamento_dados
           |____ limpeza_nulos.py
           |____ limpeza_preenchimento.py
           |____ preenche_valores.py
           |____ valores_extremos.py
           |____ categorizacao.py
     |____ modelo
           |____ classificador_xgb.py
           |____ logistica_simples.py
     |____ avaliacao
           |____ metricas_avaliacao.py
           |____ comparativo_modelos.py
     | main.py
|____ tests
|____ venv
```

#### Leitura das bases

Famílias de Métodos: <br>
**scan_** <br>
Utilizados a leitura de um arquivo/tabela para um grafo/plano de consulta. Não é indicado para fluxos de trabalho onde seja necessário conhecer os valores dos dados. Porém, o método *.collect* pode ser oportunamente chamado quando necessário e o datafrale liberado posteriormente chamando o método *.lazy*.

**read_** <br>
Utilizados a leitura de um arquivo/tabela para um dataframe colunar. 

Formatos de arquivos e outros
- Parquet
- CSV
- Avro
- Delta
- JSON

Interoperabilidade com outras engines/bibliotecas:
**Pandas** <br>

Um dataframe Polars pode ser convertido de maneira simples. Aplicar o método *.to_pandas()* converte a o dataframe Polars em um dataframe Pandas. Alguns métodos do polars fazem a serialização para tipos numpy para calcular algumas funções estatísticas

```python
import polars as pl
import duckdb as dd

df = pl.DataFrame({"foo": [1, 2, 3], "bar": [4, 5, 6]})
pandas_df = df.to_pandas()
print(pandas_df)
``` 

**Duckdb** <br>
Utilizar um dataframe Polars com o DuckDB é extremamente simples, uma vez que este pode ser acessado de forma similar ao acessar uma tabela em um banco de dados convencional.

```python
import polars as pl
import duckdb as dd

df = pl.DataFrame({"foo": [1, 2, 3], "bar": [4, 5, 6]})
dd.sql(
    """
        select * from df
    """
).show()
```
**Pyspark**<br>
É pouco provável precisar deste tipo de conversão, mas uma forma possível e não performática é converter o dataframe em uma lista de dicionários, iterar sobre os seus itens e construir um dataframe Spark através do método createDataFrame
```python

import polars as pl
from pyspark.sql import SparkSession

df = pl.DataFrame({"foo": [1, 2, 3], "bar": [4, 5, 6]})

os.environ["SPARK_HOME"] = "D:/spark"
os.environ["HADOOP_HOME"] = "D:/spark/hadoop"

spark = SparkSession.builder.appName("Teste").getOrCreate()
spark.createDataFrame({x:y.to_list() for x,y in df.to_dict().items()})
```

Outras situações importantes podem envolver transformar um dataframe em componentes serializáveis (dicionários) que podem posteriormente serem convertidos em json e transitados via uma API HTTP.

```python
df = pl.DataFrame({"foo": [1, 2, 3], "bar": [4, 5, 6]})
lista_dicionarios = df.to_dicts()

print(lista_dicionarios)
```

In [7]:
caminho_base = Path("C:/Users/RenatoFileto/curso_bigdata/ModuloPython/data/raw")
caminho_base_csv = caminho_base.joinpath("base_semana02.csv")

grafo_base_atributos = pl.scan_csv(
    caminho_base_csv,
    low_memory=True,
    separator=";",
    has_header=True,
    infer_schema_length=100000,
    cache=False,
)


#### Estratégias de sink e collect


In [15]:
dicionario_meses_pt_br = {
    "jan": 1,
    "fev": 2,
    "mai": 5,
    "jul": 7,
    "set": 9,
    "dez": 12,
}

grafo_base_atributos = (
    grafo_base_atributos.select(
        "foto",
        split1=pl.col("foto").str.extract_all(r"\D+").list.first(),
        split2=pl.col("foto").str.extract_all(r"\d+").list.first(),
    )
    .with_columns(trat_foto=pl.col("split1").str.to_lowercase().str.strip_chars(" "))
    .with_columns(
        trat_mes=pl.col("trat_foto").replace(dicionario_meses_pt_br),
        trat_ano=(pl.col("split2").cast(pl.Int16) + 2000),
    )
    .with_columns(
        trat_data_ref=pl.date(pl.col("trat_ano"), pl.col("trat_mes"), 1).dt.month_end()
    )
)

lista_datas_referencia: pl.Series = (
    grafo_base_atributos.select("trat_data_ref").unique().collect().to_series()
)

print(lista_datas_referencia)

shape: (6,)
Series: 'trat_data_ref' [date]
[
	2022-07-31
	2023-02-28
	2022-05-31
	2022-09-30
	2022-02-28
	2021-12-31
]


#### Remoção de colunas e linhas nulas

Métodos:
- **drop_null**

#### Inputação de valores

Métodos Polars:
- **fill_null** e estratégias *foward*, *backward* e *min, max, mean e median*
- **fill_nan**

Existem outras formas e métodos mais sofisticados de lidar com estes valores.  Uma delas é aplicar um algoritmo K-Nearest Neighbors e utilizar a média dos valores do vizinhos. Também é possível utilizar um algoritmo de regressão para determianr o valor a ser preenchido.


#### Remoção de colunas que possuam poucos elementos distintos

#### Tratamento de valores extremos

#### Descritiva simples



#### Categoricação de covariáveis

1. qcut
2. cut

#### (Bônus) 

- Plotagem
[Seaborn](https://www.kaggle.com/code/themlphdstudent/cheat-sheet-seaborn-charts)

- Visualização
[NiceGUI](https://nicegui.io/documentation)